# Hangman — training (Kaggle GPU)

Trains the masked-word transformer with on-policy state aggregation.

**Turn the GPU accelerator on. Internet can stay off — nothing here downloads anything.**

Round 0 collects board states by playing every training word with the n-gram
heuristic. Each later round replays with the model trained so far, aggregates the
new states into the buffer, and retrains from scratch on the union. Two or three
rounds is normally where the holdout win rate stops moving.

Numbers to beat, measured on this exact data:

| policy | holdout (10k) | test.txt (250k) |
|---|---|---|
| length-conditioned frequency | 13.03% | — |
| character n-gram back-off | 48.50% | **50.80%** |


In [ ]:
import os, sys, time, json, warnings
warnings.filterwarnings("ignore")

# The package is attached as a Kaggle Dataset (or unzipped into /kaggle/working).
sys.path.insert(0, "/kaggle/input/hangman-src/src")
sys.path.insert(0, "/kaggle/working/src")

import numpy as np
import torch

from hangman.data import (load_words, audit, overlap, split_holdout,
                          find_competition_dir)
from hangman.policies import LengthFrequencyPolicy, NeuralPolicy, EpsilonMixPolicy
from hangman.policies.ngram import NGramPolicy
from hangman.states import collect, BoardBatcher, StateBuffer
from hangman.model import HangmanNet, ModelConfig
from hangman.train import train, TrainConfig
from hangman.evaluate import evaluate, print_report

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 1. Data audit

Expected: 225,300 train words, 250,000 test words, lengths 1–29, pure `a–z`,
and **zero** test words present in train.

In [ ]:
COMP = find_competition_dir()
print("data:", COMP)
train_words_all = load_words(f"{COMP}/train.txt")
test_words = load_words(f"{COMP}/test.txt")

print(audit(train_words_all)); print()
print(audit(test_words)); print()
print("overlap:", overlap(train_words_all, test_words))

MAX_LEN = max(max(map(len, train_words_all)), max(map(len, test_words)))
print("MAX_LEN =", MAX_LEN)

## 2. Holdout split and the heuristic policy

`test.txt` is a validation list we are not allowed to train on, so the honest
signal comes from words held out of `train.txt`. The n-gram policy is both the
baseline to beat and the generator that bootstraps round 0.

In [ ]:
TRAIN_WORDS, HOLDOUT = split_holdout(train_words_all, n_holdout=10_000, seed=0)
print(len(TRAIN_WORDS), len(HOLDOUT))

heuristic = NGramPolicy(TRAIN_WORDS)     # ~25s to build
freq = LengthFrequencyPolicy(TRAIN_WORDS)
print_report(evaluate(HOLDOUT, heuristic, max_len=MAX_LEN))

## 3. Round 0 — bootstrap states from the heuristic

A little exploration widens the state distribution without wrecking it.
`epsilon=0.02, noise=0.05` costs the generator about five points of win rate,
which is the right trade; stronger settings collapse it (measured: `noise=0.15`
took the n-gram policy from 54% to 30%).

In [ ]:
generator = EpsilonMixPolicy(heuristic, freq, epsilon=0.02, noise=0.05, seed=1)

t = time.time()
buffer, stats = collect(TRAIN_WORDS, generator, max_len=MAX_LEN)   # ~6 min
print(f"{len(buffer):,} states in {time.time()-t:.0f}s   generator: {stats}")
buffer.save("/kaggle/working/states_round0.npz")

## 4. DAgger rounds

Keep every round's states; retrain from scratch each time so the model does not
inherit the previous policy's blind spots.

In [ ]:
MODEL_CFG = ModelConfig(d_model=256, n_layers=8, n_heads=8, d_ff=1024,
                        dropout=0.1, max_len=MAX_LEN)
ROUNDS = 3
history = []

for rnd in range(ROUNDS):
    batcher = BoardBatcher(TRAIN_WORDS, buffer, max_len=MAX_LEN,
                           batch_size=512, bucket=True, seed=rnd)
    cfg = TrainConfig(epochs=3, batch_size=512, lr=3e-4,
                      amp=(DEVICE == "cuda"), model=MODEL_CFG, seed=rnd)
    print(f"\n=== round {rnd}: {len(buffer):,} states, {len(batcher):,} batches/epoch ===")
    model = train(batcher, cfg, device=DEVICE)

    torch.save({"model": model.state_dict(), "cfg": MODEL_CFG.__dict__},
               f"/kaggle/working/hangman_r{rnd}.pt")

    policy = NeuralPolicy(model, DEVICE, fusion=0.5)
    metrics = evaluate(HOLDOUT, policy, max_len=MAX_LEN)
    print_report(metrics)
    history.append({"round": rnd, "win_rate": metrics["win_rate"],
                    "mean_wrong": metrics["mean_wrong"], "states": len(buffer)})
    json.dump(history, open("/kaggle/working/history.json", "w"), indent=2)

    if rnd + 1 < ROUNDS:
        explore = EpsilonMixPolicy(policy, heuristic, epsilon=0.02, noise=0.05,
                                   seed=100 + rnd)
        new_states, stats = collect(TRAIN_WORDS, explore, max_len=MAX_LEN)
        print(f"collected {len(new_states):,} new states; {stats}")
        buffer = StateBuffer.concat([buffer, new_states])
        buffer.save(f"/kaggle/working/states_round{rnd+1}.npz")

history

## 5. Tune the head-fusion weight on the holdout

`fusion=1.0` uses the noisy-OR aggregation of the per-position MLM head alone;
`fusion=0.0` uses the pooled bag-of-letters head alone.

In [ ]:
best = (None, -1.0)
for fusion in np.linspace(0.0, 1.0, 11):
    m = evaluate(HOLDOUT[:4000], NeuralPolicy(model, DEVICE, fusion=float(fusion)),
                 max_len=MAX_LEN)
    print(f"fusion={fusion:.1f}  win {m['win_rate']:.2f}%  strikes {m['mean_wrong']:.3f}")
    if m["win_rate"] > best[1]:
        best = (float(fusion), m["win_rate"])
print("best fusion:", best)
json.dump({"fusion": best[0]}, open("/kaggle/working/inference.json", "w"))

Save `hangman_r*.pt` and `inference.json` as a Kaggle Dataset, then attach that
dataset to `02_submit.ipynb`.